# Buffered Loading with Shuffle Buffer

**Difficulty**: 🟡 Medium

---

### Problem Statement

`IterableDataset` streams data sequentially, which means samples arrive in file order — **no shuffling**. `DataLoader(shuffle=True)` only works with map-style datasets.

The standard solution is a **shuffle buffer**: accumulate `B` samples in a buffer, then yield one at random from the buffer, replacing it with the next incoming sample. This gives approximate shuffling with O(B) memory, regardless of dataset size. This is exactly how TensorFlow's `tf.data.Dataset.shuffle(buffer_size)` works, and how HuggingFace `datasets` streams shuffled data.

Your task is to implement a shuffle buffer on top of a streaming dataset.

---

### Requirements

1. **`ShuffleBufferDataset(base_dataset, buffer_size)`** — An `IterableDataset` that:
   - Wraps another `IterableDataset`.
   - Fills a buffer of `buffer_size` samples from the base iterator.
   - On each `yield`: picks a random index in the buffer, yields that sample, and replaces it with the next sample from the base iterator.
   - When the base iterator is exhausted, drains the remaining buffer in random order.

2. **Verify approximate shuffling** — Show that the output order differs from the input order.

3. **Verify completeness** — All samples from the base dataset must appear exactly once.

---

### Constraints

- ✅ Memory usage must be O(buffer_size), not O(N).
- ✅ Every sample from the base dataset must be yielded exactly once.
- ✅ Must work with `DataLoader` batching.
- ✅ Larger buffer → better shuffling (buffer_size=N is equivalent to full shuffle).

---

<details>
  <summary>💡 Hint</summary>

  - Initialize the buffer by pulling `buffer_size` items from the base iterator using `next()` in a loop.
  - Use `random.randint(0, len(buffer)-1)` to pick the index to yield.
  - Swap the yielded item with the next item from the base iterator. When the base iterator raises `StopIteration`, start draining.
  - To drain: shuffle the remaining buffer and yield everything.

</details>

---

In [ ]:
import torch
import random
from torch.utils.data import IterableDataset, DataLoader

# A simple sequential IterableDataset to wrap
class SequentialDataset(IterableDataset):
    """Yields (index_tensor, value_tensor) pairs in order 0..N-1."""
    def __init__(self, N: int):
        self.N = N
    def __iter__(self):
        for i in range(self.N):
            yield torch.tensor(i, dtype=torch.long), torch.tensor(float(i))

# Test it
seq_ds = SequentialDataset(20)
print("Sequential order:", [idx.item() for idx, _ in seq_ds])

In [ ]:
class ShuffleBufferDataset(IterableDataset):
    """Wraps an IterableDataset and shuffles its output via a fixed-size buffer."""

    def __init__(self, base_dataset: IterableDataset, buffer_size: int):
        super().__init__()
        self.base_dataset = base_dataset
        self.buffer_size = buffer_size

    def __iter__(self):
        # TODO: Get an iterator from the base dataset

        # TODO: Fill the buffer with buffer_size samples from the iterator

        # TODO: While the base iterator has more samples:
        #   - Pick a random index in the buffer
        #   - Yield the sample at that index
        #   - Replace it with the next sample from the iterator

        # TODO: Drain the remaining buffer in random order

        ...

In [ ]:
# === Tests ===

N = 500

# Test 1: All samples present, no duplicates
shuffled_ds = ShuffleBufferDataset(SequentialDataset(N), buffer_size=50)
indices = sorted([idx.item() for idx, _ in shuffled_ds])
assert indices == list(range(N)), "Not all samples yielded or duplicates detected"
print(f"PASSED: All {N} samples present, no duplicates")

# Test 2: Order is shuffled (not sequential)
shuffled_ds2 = ShuffleBufferDataset(SequentialDataset(N), buffer_size=50)
order = [idx.item() for idx, _ in shuffled_ds2]
assert order != list(range(N)), "Output is still in sequential order — not shuffled"
print("PASSED: Output order is shuffled")

# Test 3: Works with DataLoader
shuffled_ds3 = ShuffleBufferDataset(SequentialDataset(N), buffer_size=100)
loader = DataLoader(shuffled_ds3, batch_size=32)
total = sum(bi.shape[0] for bi, _ in loader)
assert total == N, f"DataLoader yielded {total} samples, expected {N}"
print("PASSED: Works with DataLoader batching")

# Test 4: Larger buffer → more shuffling
# Measure how far each element moved from its original position
def avg_displacement(ds):
    order = [idx.item() for idx, _ in ds]
    return sum(abs(i - order[i]) for i in range(len(order))) / len(order)

disp_small = avg_displacement(ShuffleBufferDataset(SequentialDataset(N), buffer_size=10))
disp_large = avg_displacement(ShuffleBufferDataset(SequentialDataset(N), buffer_size=200))
print(f"Avg displacement: buffer=10 → {disp_small:.1f}, buffer=200 → {disp_large:.1f}")
assert disp_large > disp_small, "Larger buffer should give more shuffling"
print("PASSED: Larger buffer produces more shuffling")

print("\nAll tests passed!")